# 3D-FY: AI Driven 2D to 3D Converter
### Converts 2D video into Anaglyph, Side-by-Side VR, and Depth Map outputs
**Model:** MiDaS DPT-Hybrid | **GPU:** Kaggle T4 

In [12]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/mohiadiy/avatar/VID_20260407_162155.mp4


# Install Dependencies

In [37]:
!pip install gradio==3.50.2 -q
!pip install timm gradio -q
print("installations comeplete")

installations comeplete


timm is a library of pretrained vision models and midas internally uses it

# Imports

In [27]:
# imports
import torch
import cv2
import numpy as np
import os

print("imports succesful")

imports succesful


# Settings 

In [28]:
VIDEO_PATH = "/kaggle/input/datasets/mohiadiy/avatar/VID_20260407_162155.mp4"   # your uploaded video path
Max_sec = 30  # how many seconds of video to process
#output selection
print("Select Output Type")
print("1-> for Anaglyph Output (Red-Cyan Glasses)")
print("2-> for SBS Output (VR Headset)")
print("3-> for Depth Map Output Only")
print("4-> for All 3 Outputs")

Output= input("\n Enter Choice (1/2/3/4) : ").strip()
if Output not in ["1","2","3","4"]:
    print("Invalid Choice, Defaulting to 4 (All Outputs)")
    Output= "4"

type_names= {"1": "Anaglyph", "2": "SBS", "3": "Depth Map only", "4": "All 3 Outputs"}

print(f"\n SELECTED : {type_names[Output]}")

Select Output Type
1-> for Anaglyph Output (Red-Cyan Glasses)
2-> for SBS Output (VR Headset)
3-> for Depth Map Output Only
4-> for All 3 Outputs



 Enter Choice (1/2/3/4) :  4



 SELECTED : All 3 Outputs


* paste the copy path of input video in Video_path.
* our all outputs will be saved into Output_folder i.e. Kaggle output directory.
* Strength is how far pixel should shift, higher value= stronger 3d effect but also some distortion.
* MAX_SECONDS: how long your input should be, it trim the input video upto that specific seconds only.
* Select the Output type by choosing a number for 1- Anaglyph, 2- SBS, 3- Depth Map, 4- All, any other number- default 4


In [29]:
Output_folder = "/kaggle/working/output"
os.makedirs(Output_folder, exist_ok= True)
device= "cuda" if torch.cuda.is_available() else "cpu"
print("using device: ", device)

using device:  cuda


# Load MiDaS

In [30]:
#load midas
model= torch.hub.load("intel-isl/MiDaS", "DPT_Hybrid", trust_repo= True)
model.to(device)
model.eval()

transforms= torch.hub.load("intel-isl/MiDaS", "transforms", trust_repo= True)
transform= transforms.dpt_transform
print("Model loaded on: ", device)

Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master
/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name vit_base_resnet50_384 to current vit_base_r50_s16_384.orig_in21k_ft_in1k.
  model = create_fn(


Model loaded on:  cuda


Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master


* MiDaS is a pytorch model developed by intel, we are using DPT_Hybrid version here
* model.eval() turns model into inference mode from training mode

# The Depth Estimation Function

In [31]:
#the depth function
Strength = 20      # 3D effect intensity (try 15 to 40)
prev_depth= None
smooth= 0.75

def get_depth(frame):
    global prev_depth

    small= cv2.resize(frame, (640, 480))
    rgb= cv2.cvtColor(small, cv2.COLOR_BGR2RGB)
    input_tensor= transform(rgb).to(device)
    #inferencing input frame through model
    with torch.no_grad():  #only inference
        depth= model(input_tensor) 
        depth= torch.nn.functional.interpolate( #function to resize tensors
            depth.unsqueeze(1),  #adds dimension
            size= frame.shape[:2],
            mode= 'bicubic',
            align_corners= False
        ).squeeze().cpu().numpy()
    #normalization of frames in 0-1
    depth= (depth-depth.min())/ (depth.max()- depth.min()+ 1e-8)
    if prev_depth is not None:
        depth= (smooth * depth) + ((1- smooth) * prev_depth)
    prev_depth= depth
    return depth
    



* here we are inferencing our frame from input video through the model, but before we are resizing the frame suitable for our midas model like resizing, converting BGR into RGB, adding dimensions, and most importantly converting our input frame into a tensor
* the normalization function scales all values of the tensor array to 0-1, The 1e-8 (a tiny number) is added to the denominator to prevent division by zero on completely flat scenes.
* The smoothing block is temporal smoothing. On the first frame prev_depth is None so we skip it. From second frame onward, we blend current depth with previous depth using the SMOOTH weight. 0.75 means 75% current frame, 25% previous frame. it prevents frames from flickering.

# 3D Output Function

In [32]:
# pixel shifting
def shift_pixels(frame, depth, direction):
    h,w= frame.shape[:2]
    shift= (depth* Strength* direction).astype(np.int32)
    ys, xs= np.mgrid[0:h, 0:w]
    new_xs= np.clip(xs + shift, 0, w-1)
    out= np.zeros_like(frame)
    out[ys, new_xs]= frame[ys, xs]
    return out

def make_anaglyph(frame, depth):
    left= shift_pixels(frame, depth, +1)
    right= shift_pixels(frame, depth, -1)
    anaglyph= np.zeros_like(frame)
    anaglyph[:,:,2]= left[:,:,2]
    anaglyph[:,:,1]= right[:,:, 1]
    anaglyph[:,:,0]= right[:,:, 0]
    return anaglyph

def make_sbs(frame, depth):
    left= shift_pixels(frame, depth, +1)
    right= shift_pixels(frame, depth, -1)
    return np.concatenate([left, right], axis=1)

* for every pixel multiply its depth value (0-1) by strength (mentioned in settings) to get how many pixels to shift
* np.mgrid[0:h, 0:w]: creates two 2d grids, one contains y coordinate of every pixel, one contains x coordinate of every pixel
* new_xs = np.clip(xs + shifts, 0, w-1) adds the shift to every x coordinate simultaneously. clip ensures no coordinate goes out of bounds — beyond left or right edge of the image.
* For make_sbs — np.concatenate([left, right], axis=1) joins two arrays side by side. axis=1 means concatenate along the width dimension (horizontal). The result has double the width.

# Main Loop
## Step 7 — Option A: Manual Processing Loop
> Run this cell if you want direct output files without the Gradio interface.
> Skip this and run Option B below if you want the interactive web interface.

In [22]:
cap= cv2.VideoCapture(VIDEO_PATH)
fps= cap.get(cv2.CAP_PROP_FPS)
w= int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h= int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
max_frames= int(fps* Max_sec)

print(f"Video: {w}*{h} at {fps: .0f} FPS| Processing {max_frames} frames")
print(f"\n Processing {max_frames} frames  - Output {type_names[Output]}")

fourcc= cv2.VideoWriter_fourcc(*'mp4v')

out_ana= cv2.VideoWriter(f"{Output_folder}/anaglyph.mp4", fourcc, fps, (w,h))
out_sbs= cv2.VideoWriter(f"{Output_folder}/sidebyside.mp4", fourcc, fps, (w*2,h))
out_dep= cv2.VideoWriter(f"{Output_folder}/depthmap.mp4", fourcc, fps, (w,h))

frame_num= 0
while frame_num< max_frames:
    ret, frame = cap.read()
    if not ret:
        break

    depth= get_depth(frame)
    anaglyph= make_anaglyph(frame, depth)
    sbs= make_sbs(frame, depth)
    depth_colored= cv2.applyColorMap((depth*255).astype(np.uint8), cv2.COLORMAP_MAGMA)
   
    out_ana.write(anaglyph)
    out_sbs.write(sbs)
    out_dep.write(depth_colored)


    frame_num+=1
    if frame_num%20== 0:
        print(f"Processed {frame_num}/ {max_frames} frames")

cap.release()

if out_ana is not None: out_ana.release()
if out_sbs is not None: out_sbs.release()
if out_dep is not None: out_dep.release()


if Output in ["1", "4"]: print(f"  Anaglyph  → {Output_folder}/anaglyph.mp4")
if Output in ["2", "4"]: print(f"  SBS       → {Output_folder}/sidebyside.mp4")
if Output in ["3", "4"]: print(f"  Depth map → {Output_folder}/depthmap.mp4")




def add_audio(original_video, silent_video, output_path):
    cmd = (
        f'ffmpeg -i "{silent_video}" -i "{original_video}" '
        f'-c:v copy -c:a aac -map 0:v:0 -map 1:a:0 '
        f'-shortest "{output_path}" -y -loglevel error'
    )
    os.system(cmd)
    print(f"Audio added → {output_path}")

add_audio(VIDEO_PATH, f"{Output_folder}/anaglyph.mp4",   f"{Output_folder}/anaglyph_final.mp4")
add_audio(VIDEO_PATH, f"{Output_folder}/sidebyside.mp4", f"{Output_folder}/sidebyside_final.mp4")
add_audio(VIDEO_PATH, f"{Output_folder}/depthmap.mp4",  f"{Output_folder}/depth_final.mp4")

Video: 1920*1080 at  24 FPS| Processing 720 frames

 Processing 720 frames  - Output All 3 Outputs
Processed 20/ 720 frames
Processed 40/ 720 frames
Processed 60/ 720 frames
Processed 80/ 720 frames
Processed 100/ 720 frames
Processed 120/ 720 frames
Processed 140/ 720 frames
Processed 160/ 720 frames
Processed 180/ 720 frames
Processed 200/ 720 frames
Processed 220/ 720 frames
Processed 240/ 720 frames
Processed 260/ 720 frames
Processed 280/ 720 frames
Processed 300/ 720 frames
Processed 320/ 720 frames
Processed 340/ 720 frames
Processed 360/ 720 frames
Processed 380/ 720 frames
Processed 400/ 720 frames
Processed 420/ 720 frames
Processed 440/ 720 frames
Processed 460/ 720 frames
Processed 480/ 720 frames
Processed 500/ 720 frames
Processed 520/ 720 frames
Processed 540/ 720 frames
Processed 560/ 720 frames
Processed 580/ 720 frames
Processed 600/ 720 frames
Processed 620/ 720 frames
Processed 640/ 720 frames
Processed 660/ 720 frames
Processed 680/ 720 frames
Processed 700/ 720 fr

* cv2.VideoWriter_fourcc(*'mp4v') defines the video codec — the compression algorithm for the output file. mp4v is standard MPEG-4 encoding compatible with most players. The * unpacks the string into four separate characters as the function expects
* cv2.VideoWriter(path, fourcc, fps, (width, height)) creates an output video file. Notice SBS gets (w*2, h) because it's double width. If this size doesn't match the frames you write to it, OpenCV silently drops those frames — no error, just missing frames in output. This is a common silent bug.

# Gradio Interface
## Step 7 — Option B: Gradio Web Interface  
> Run this cell for an interactive interface with a public shareable link.
> Skip the manual loop above if running this.

In [41]:
import nest_asyncio
nest_asyncio.apply()

import gradio as gr
import tempfile

def process(input_video, choice, strength, max_sec):
    global prev_depth
    prev_depth  = None
    strength    = int(strength)
    max_sec     = int(max_sec)

    if input_video is None:
        return None, None, None

    out_dir    = tempfile.mkdtemp()
    fourcc     = cv2.VideoWriter_fourcc(*'mp4v')

    cap        = cv2.VideoCapture(input_video)
    fps        = cap.get(cv2.CAP_PROP_FPS) or 24
    w          = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h          = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    max_frames = int(fps * max_sec)

    if w == 0 or h == 0:
        return None, None, None

    ana_path = os.path.join(out_dir, "anaglyph.mp4")
    sbs_path = os.path.join(out_dir, "sidebyside.mp4")
    dep_path = os.path.join(out_dir, "depth.mp4")

    out_ana = cv2.VideoWriter(ana_path, fourcc, fps, (w, h))   if choice in ["Anaglyph",    "All"] else None
    out_sbs = cv2.VideoWriter(sbs_path, fourcc, fps, (w*2, h)) if choice in ["Side-by-Side", "All"] else None
    out_dep = cv2.VideoWriter(dep_path, fourcc, fps, (w, h))   if choice in ["Depth Map",    "All"] else None

    frame_num = 0
    while frame_num < max_frames:
        ret, frame = cap.read()
        if not ret:
            break

        depth = get_depth(frame)

        if out_ana is not None:
            out_ana.write(make_anaglyph(frame, depth, strength))
        if out_sbs is not None:
            out_sbs.write(make_sbs(frame, depth, strength))
        if out_dep is not None:
            out_dep.write(cv2.applyColorMap(
                (depth * 255).astype(np.uint8), cv2.COLORMAP_MAGMA
            ))

        frame_num += 1

    cap.release()
    if out_ana is not None: out_ana.release()
    if out_sbs is not None: out_sbs.release()
    if out_dep is not None: out_dep.release()

    def remux(silent, original, output):
        cmd = (
            f'ffmpeg -y -i "{silent}" -i "{original}" '
            f'-c:v libx264 -preset fast -crf 23 '
            f'-c:a aac -map 0:v:0 -map 1:a:0 '
            f'-shortest "{output}" -loglevel error'
        )
        os.system(cmd)
        return output if os.path.exists(output) else None

    final_ana = remux(ana_path, input_video, os.path.join(out_dir, "anaglyph_final.mp4")) if out_ana is not None else None
    final_sbs = remux(sbs_path, input_video, os.path.join(out_dir, "sbs_final.mp4"))      if out_sbs is not None else None
    final_dep = remux(dep_path, input_video, os.path.join(out_dir, "depth_final.mp4"))    if out_dep is not None else None

    return final_ana, final_sbs, final_dep


with gr.Blocks(title="3D-FY", theme=gr.themes.Soft()) as app:

    gr.Markdown("# 🎬 3D-FY — AI 2D to 3D Video Converter")
    gr.Markdown("Powered by MiDaS Depth Estimation · Kaggle T4 GPU · For VR headsets and 3D glasses")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### ⚙️ Settings")
            input_video = gr.File(
                label="Upload your 2D video (.mp4)",
                file_types=[".mp4", ".avi", ".mov", ".mkv"]
            )
            choice   = gr.Dropdown(
                         choices=["Anaglyph", "Side-by-Side", "Depth Map", "All"],
                         value="All",
                         label="Output Format"
                       )
            strength = gr.Slider(minimum=5, maximum=50, value=25, step=5, label="3D Effect Strength")
            max_sec  = gr.Slider(minimum=5, maximum=60, value=15, step=5, label="Max Seconds to Process")
            btn      = gr.Button("🚀 Convert to 3D", variant="primary")

        with gr.Column():
            gr.Markdown("### 📤 Outputs")
            out_ana_widget = gr.Video(label="Anaglyph — Red Cyan Glasses")
            out_sbs_widget = gr.Video(label="Side-by-Side — VR Headset")
            out_dep_widget = gr.Video(label="Depth Map Visualization")

    gr.Markdown("""
    ---
    **How to use:** Upload .mp4 → Select format → Adjust strength → Click Convert

    **Formats:** Anaglyph = red-cyan glasses · Side-by-Side = VR headset · Depth Map = AI visualization

    **Project:** 3D-FY | JB Institute of Technology Dehradun | MiDaS DPT-Hybrid on Kaggle T4 GPU
    """)

    btn.click(
        fn=process,
        inputs=[input_video, choice, strength, max_sec],
        outputs=[out_ana_widget, out_sbs_widget, out_dep_widget]
    )

app.launch(share=True, debug=True, show_error=True)

/tmp/ipykernel_57/597342264.py:77: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="3D-FY", theme=gr.themes.Soft()) as app:
/usr/local/lib/python3.12/dist-packages/gradio/analytics.py:93: UserWarning: IMPORTANT: You are using gradio version 3.50.2, however version 4.44.1 is available, please upgrade. 
--------
  if StrictVersion(latest_pkg_version) > StrictVersion(current_pkg_version):


* Running on local URL:  http://127.0.0.1:7866
* Running on public URL: https://efe161fd8044aea9c2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://270dabf6ce2dfe085d.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://6d8700fa4753225ad5.gradio.live
Killing tunnel 127.0.0.1:7862 <> https://81a8c5735aa91065e9.gradio.live
Killing tunnel 127.0.0.1:7863 <> https://302f070e0434d8912d.gradio.live
Killing tunnel 127.0.0.1:7864 <> https://7a1970839818cf92a0.gradio.live
Killing tunnel 127.0.0.1:7865 <> https://d7fb3cd8d5c8573a5e.gradio.live
Killing tunnel 127.0.0.1:7866 <> https://efe161fd8044aea9c2.gradio.live
